In [7]:
# Resolve paths consistently when launched from the root or notebooks/.
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "topic_embeddings").is_dir() and (p / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch this notebook from within the derStandard project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)


# Topic embeddings with SGNS

Learn topic embeddings from the top 1,000 leaf topics selected in the previous notebook. In [word2vecf](https://github.com/BIU-NLP/word2vecf), topics are **words** and authors are **contexts**. Each retained comment contributes one topic–author pair, including repeated pairs. The comment text isn't used.

The current settings are 150 dimensions, 35 negative samples, a starting learning rate of 0.18, subsampling at 0.0043, and one pass with four threads. Negative authors are sampled based on their comment counts raised to 0.75. Subsampling applies to both topics and authors.

Comments are read in chunks and the pairs are shuffled globally before training. `SHUFFLE_SEED` controls that shuffle; it doesn't control the trainer's weight initialization. Training with multiple threads can still give slightly different results between runs.


In [8]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

DATA_DIR = Path("data/processed")
RAW_DATA_DIR = Path("data/raw")
OUTPUT_DIR = Path("results/sgns")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
USE_RAW_COMMENTS = False
TOPIC_COLUMNS = ["Topic_1", "Topic_2", "Topic_3"]
CHUNK_SIZE = 250_000
SHUFFLE_SEED = 42

# word2vecf hyperparameters
DIMENSIONS = 150
NEGATIVE = 35
ALPHA = 0.075
SAMPLE = 0.001 # subsampling threshold
EPOCHS = 1
THREADS = 4

from topic_embeddings.sgns import build_trainer, train_embeddings, export_embeddings


## 1. Build word2vecf

Clone word2vecf if it isn't already in `.tools/word2vecf`, check that it matches `TOOL_COMMIT`, and compile it with `cc` into `OUTPUT_DIR`. This needs Git and a C compiler. Compiler output and errors are saved to `results/sgns/build.log`.


In [9]:
trainer = build_trainer(OUTPUT_DIR)

## 2. Load selected topics, articles and comments

Read `top_1000_topics.csv` and `top_1000_articles.csv` from `data/processed` and strip whitespace from the topic labels. 

`comment_chunks()` reads  `top_1000_comments.csv`, in batches of 250,000 rows. With `USE_RAW_COMMENTS = False`, it reads `data/processed/top_1000_comments.csv`. Setting it to `True` reads the original comment TSV files in `data/raw` instead.


In [10]:
# Load topics
topics_df = pd.read_csv(DATA_DIR / "top_1000_topics.csv", keep_default_na=False,
                              dtype={column: "string" for column in TOPIC_COLUMNS})
topics_df[TOPIC_COLUMNS] = topics_df[TOPIC_COLUMNS].apply(lambda s: s.str.strip())

# Load articles and create a mapping from article ID to topic ID
articles_df = pd.read_csv(DATA_DIR / "top_1000_articles.csv", dtype="string",
                          keep_default_na=False, usecols=["Article_Id", "Topic_Id", *TOPIC_COLUMNS])
articles_df[TOPIC_COLUMNS] = articles_df[TOPIC_COLUMNS].apply(lambda s: s.str.strip())
article_to_topic = (articles_df.set_index("Article_Id")["Topic_Id"])

# Comment chunks generator
def comment_chunks():
    if USE_RAW_COMMENTS:
        paths = sorted(RAW_DATA_DIR.glob("comments_file_*_[0-9][0-9].tsv"))
        if not paths:
            raise FileNotFoundError("No raw comment shards found")
    else:
        paths = [DATA_DIR / "top_1000_comments.csv"]
    for path in paths:
        with pd.read_csv(path, sep="\t" if USE_RAW_COMMENTS else ",", dtype="string",
                         keep_default_na=False, usecols=["Article_Id", "Author_Id"],
                         chunksize=CHUNK_SIZE) as reader:
            yield from reader

## 3. Create the pairs, shuffle, and train

The trainer writes the topic vectors to `topic_vectors.txt` and its output and errors to `training.log`. `training_stats` keeps the pair count, number of authors, excluded blank-author count, and comment counts per topic for the metadata saved below. These counts are measured before the trainer's subsampling. Training returns only statistics. Vectors and internal topic and author mappings are saved to disk; the export step needs only `OUTPUT_DIR`, even after a kernel restart.


In [11]:
training_stats = train_embeddings(
    comment_chunks(), topics_df, article_to_topic, trainer, OUTPUT_DIR,
    dimensions=DIMENSIONS, negative=NEGATIVE, alpha=ALPHA, sample=SAMPLE,
    epochs=EPOCHS, threads=THREADS, shuffle_seed=SHUFFLE_SEED, chunk_size=CHUNK_SIZE,
)

## 4. Load and save the topic and user embeddings

This section writes five files to `results/sgns`:

- `topic_embeddings.csv`: topic IDs, topic labels, and the raw embedding coordinates.
- `topic_embeddings_normalized.csv`: the same IDs and labels with unit-length vectors.
- `user_embeddings.csv`: original author IDs and raw user embedding coordinates.
- `user_embeddings_normalized.csv`: the same authors with unit-length vectors (zero vectors stay zero).
- `run_metadata.json`: hyperparameters, shuffle seed, selection description, and training counts.

The earlier sections also leave these files in the same folder:

- `topic_vectors.txt`: the trainer's original topic-vector output, read by this section.
- `user_vectors.txt`: author/context vectors exported by the trainer.
- `user_mapping.csv` and `topic_mapping.csv`: original IDs and topic labels, loaded automatically by `export_embeddings(OUTPUT_DIR)`.
- `build.log`: compiler output and errors.
- `training.log`: trainer output and errors.


In [14]:
# Export the embeddings to a CSV file, raw and normalized
export_embeddings(OUTPUT_DIR)

# Save run metadata to a JSON file for reproducibility
run_metadata = dict(
    dimensions=DIMENSIONS, negative=NEGATIVE, alpha=ALPHA, sample=SAMPLE,
    epochs=EPOCHS, threads=THREADS, shuffle_seed=SHUFFLE_SEED,
    selection="top 1000 leaf topics by unique users",
    **training_stats,
)
_ = (OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2) + "\n")